# Food Delivery Performance & Operational Analytics

**Submitted by:** Khushi Chaudhary  
**Project Type:** IBM Data Analytics Internship — Pure Data Analytics  
**Dataset:** Food Delivery Time Prediction (~50,000 records, 24 columns)  
**Tools:** Python · Pandas · NumPy · Plotly · Matplotlib · Seaborn  
**Note:** No Machine Learning · No Generative AI · No Prediction Models

---

## Table of Contents
1. Imports & Setup
2. Dataset Loading & Validation
3. Data Cleaning
4. Data Quality Checks
5. Feature Engineering
6. 10 KPI Calculations
7. EDA — Univariate Analysis
8. EDA — Bivariate Analysis
9. Correlation Analysis
10. Order Trend Analysis
11. BQ1 — Average Delivery Time
12. BQ2 — Delivery Time by Day of Week
13. BQ3 — Delivery Time by Time of Day
14. BQ4 — Traffic Impact on Delivery
15. BQ5 — Distance Impact on Delivery
16. BQ6 — Vehicle Type Performance
17. BQ7 — Preparation Time Impact
18. BQ8 — Rider Performance Analysis
19. BQ9 — Weather Impact
20. BQ10 — Zone Performance
21. Operational & Environmental Analysis
22. Business Insights Summary

## 1. Imports & Setup

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Display settings ──────────────────────────────────────────────────────
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['figure.dpi'] = 100

# ── Colour palette (consistent across all charts) ────────────────────────
PRIMARY   = '#3b82d4'
ACCENT    = '#e05c32'
PALETTE   = px.colors.qualitative.Plotly

print('All libraries loaded successfully.')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')

## 2. Dataset Loading & Validation

In [ ]:
# ── Resolve dataset path (works whether run from project root or notebooks/) ─
NOTEBOOK_DIR = os.path.abspath('')
# Walk up until we find data/raw/
for _candidate in [NOTEBOOK_DIR, os.path.dirname(NOTEBOOK_DIR)]:
    _csv = os.path.join(_candidate, 'data', 'raw', 'Food_Delivery_Time_Prediction.csv')
    if os.path.exists(_csv):
        CSV_PATH = _csv
        break
else:
    raise FileNotFoundError('Cannot find Food_Delivery_Time_Prediction.csv. '
                            'Place the notebook inside food_delivery_analytics/ '
                            'and ensure data/raw/ exists.')

print(f'Dataset path: {CSV_PATH}')

# ── Expected 24 columns ───────────────────────────────────────────────────
EXPECTED_COLUMNS = [
    'Order_ID','Order_Date','Order_Hour','Day_of_Week','Is_Weekend',
    'Is_Festival','Weather','Pickup_Zone','Dropoff_Zone','Vehicle_Type',
    'Rider_Experience_Years','Rider_Rating','Restaurant_Rating','Cuisine_Type',
    'Order_Items','Restaurant_Load','Preparation_Time_Min','Road_Distance_km',
    'Delivery_Distance_Category','Traffic_Level','Number_of_Signals',
    'Average_Speed_kmph','Delivery_Priority','Time_taken_min',
]

DTYPE_MAP = {
    'Order_ID': str, 'Order_Hour': 'Int64', 'Day_of_Week': str,
    'Is_Weekend': 'Int64', 'Is_Festival': 'Int64', 'Weather': str,
    'Pickup_Zone': str, 'Dropoff_Zone': str, 'Vehicle_Type': str,
    'Cuisine_Type': str, 'Restaurant_Load': str,
    'Delivery_Distance_Category': str, 'Traffic_Level': str,
    'Delivery_Priority': str, 'Order_Items': 'Int64',
    'Number_of_Signals': 'Int64', 'Preparation_Time_Min': 'Int64',
    'Time_taken_min': 'Int64', 'Rider_Experience_Years': float,
    'Rider_Rating': float, 'Restaurant_Rating': float,
    'Road_Distance_km': float, 'Average_Speed_kmph': float,
}

# ── Load ──────────────────────────────────────────────────────────────────
raw_df = pd.read_csv(CSV_PATH, dtype=DTYPE_MAP,
                     parse_dates=['Order_Date'], low_memory=False)

# ── Validate columns ──────────────────────────────────────────────────────
missing_cols = [c for c in EXPECTED_COLUMNS if c not in raw_df.columns]
assert len(missing_cols) == 0, f'Missing columns: {missing_cols}'

mem_mb = raw_df.memory_usage(deep=True).sum() / 1_048_576
print(f'Rows      : {len(raw_df):,}')
print(f'Columns   : {len(raw_df.columns)}')
print(f'Memory    : {mem_mb:.2f} MB')
print(f'Date range: {raw_df["Order_Date"].min().date()} to {raw_df["Order_Date"].max().date()}')
print('Column validation: PASSED')

In [ ]:
# ── First 5 rows ──────────────────────────────────────────────────────────
raw_df.head()

In [ ]:
# ── Column data types ─────────────────────────────────────────────────────
raw_df.dtypes.to_frame('dtype').T

## 3. Data Cleaning

In [ ]:
df = raw_df.copy()
original_rows = len(df)
cleaning_log = []

def log(msg):
    cleaning_log.append(msg)
    print(f'  {msg}')

print('=== DATA CLEANING PIPELINE ===')

# ── Step 1: Remove duplicate Order_IDs ───────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset='Order_ID')
log(f'Step 1 - Duplicates removed: {before - len(df)}')

# ── Step 2: Missing value audit and imputation ────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    log('Step 2 - No missing values detected')
else:
    log(f'Step 2 - Missing values found in {len(missing)} columns:')
    for col, cnt in missing.items():
        pct = cnt / len(df) * 100
        log(f'  {col}: {cnt} nulls ({pct:.1f}%)')
        if pct < 5.0:
            if df[col].dtype in [np.float64, float]:
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df = df.dropna(subset=[col])

# ── Step 3: Type enforcement ──────────────────────────────────────────────
int_cols   = ['Order_Hour','Order_Items','Number_of_Signals',
               'Preparation_Time_Min','Time_taken_min','Is_Weekend','Is_Festival']
float_cols = ['Rider_Experience_Years','Rider_Rating','Restaurant_Rating',
               'Road_Distance_km','Average_Speed_kmph']
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
for col in float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)
log('Step 3 - Type enforcement applied')

# ── Step 4: Range validation ──────────────────────────────────────────────
invalid_masks = {
    'Time_taken_min < 1'           : df['Time_taken_min'] < 1,
    'Road_Distance_km <= 0'        : df['Road_Distance_km'] <= 0,
    'Average_Speed_kmph <= 0'      : df['Average_Speed_kmph'] <= 0,
    'Rider_Rating outside [1,5]'   : ~df['Rider_Rating'].between(1, 5),
    'Restaurant_Rating outside[1,5]': ~df['Restaurant_Rating'].between(1, 5),
    'Order_Hour outside [0,23]'    : ~df['Order_Hour'].between(0, 23),
    'Preparation_Time_Min < 0'     : df['Preparation_Time_Min'] < 0,
}
total_invalid = 0
for reason, mask in invalid_masks.items():
    count = int(mask.sum())
    if count > 0:
        df = df[~mask]
        total_invalid += count
log(f'Step 4 - Range validation: {total_invalid} invalid rows dropped')

# ── Step 5: IQR outlier capping (winsorisation) ───────────────────────────
OUTLIER_COLS = ['Time_taken_min','Road_Distance_km','Average_Speed_kmph','Preparation_Time_Min']
for col in OUTLIER_COLS:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr    = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    before_col = df[col].copy()
    df[col] = df[col].clip(lower=lower, upper=upper)
    n_capped = (before_col != df[col]).sum()
    log(f'  {col}: capped {n_capped} values -> [{lower:.2f}, {upper:.2f}]')
log('Step 5 - Outlier capping complete')

# ── Step 6: Standardise string columns ───────────────────────────────────
STRING_COLS = ['Day_of_Week','Weather','Pickup_Zone','Dropoff_Zone',
                'Vehicle_Type','Cuisine_Type','Restaurant_Load',
                'Delivery_Distance_Category','Traffic_Level','Delivery_Priority']
for col in STRING_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()
log('Step 6 - String columns standardised (strip + title case)')

# ── Step 7: Date integrity ────────────────────────────────────────────────
bad_dates = (~df['Order_Date'].between(
    pd.Timestamp('2020-01-01'), pd.Timestamp('2030-12-31'))).sum()
if bad_dates > 0:
    df = df[df['Order_Date'].between(
        pd.Timestamp('2020-01-01'), pd.Timestamp('2030-12-31'))]
log(f'Step 7 - Date integrity: {bad_dates} out-of-range rows removed')

# ── Step 8: Binary column verification ───────────────────────────────────
for col in ['Is_Weekend','Is_Festival']:
    unique_vals = set(df[col].dropna().unique())
    status = 'OK' if unique_vals.issubset({0,1}) else f'WARNING: {unique_vals}'
    log(f'Step 8 - {col} binary check: {status}')

print()
print(f'Original rows : {original_rows:,}')
print(f'Cleaned rows  : {len(df):,}')
print(f'Rows removed  : {original_rows - len(df):,} ({(original_rows-len(df))/original_rows*100:.1f}%)')

## 4. Data Quality Checks

In [ ]:
print('=== DATA QUALITY REPORT ===')
print(f'Shape : {df.shape}')
print()

# Missing values after cleaning
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) == 0:
    print('Missing values : NONE (all columns complete)')
else:
    print('Remaining nulls:')
    print(null_counts)

print()
# Duplicate check
dups = df['Order_ID'].duplicated().sum()
print(f'Duplicate Order_IDs : {dups}')
print()

# Descriptive statistics
print('--- Descriptive Statistics (numeric columns) ---')
numeric_cols = ['Time_taken_min','Road_Distance_km','Preparation_Time_Min',
                'Average_Speed_kmph','Rider_Rating','Restaurant_Rating',
                'Rider_Experience_Years','Number_of_Signals','Order_Items']
df[numeric_cols].describe().round(2)

In [ ]:
# Categorical column value counts
cat_cols = ['Weather','Traffic_Level','Vehicle_Type','Cuisine_Type',
            'Pickup_Zone','Dropoff_Zone','Delivery_Priority',
            'Restaurant_Load','Delivery_Distance_Category']

for col in cat_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts().to_string())
    print()

## 5. Feature Engineering

In [ ]:
# ── Month (1-12) ──────────────────────────────────────────────────────────
df['Month'] = df['Order_Date'].dt.month

# ── ISO week number ───────────────────────────────────────────────────────
df['Week_Number'] = df['Order_Date'].dt.isocalendar().week.astype(int)

# ── Time of Day bins ──────────────────────────────────────────────────────
hour = df['Order_Hour'].astype(float)
conditions = [
    hour.between(6,  11),
    hour.between(12, 16),
    hour.between(17, 20),
    hour.between(21, 23),
    hour.between(0,   5),
]
choices = ['Morning','Afternoon','Evening','Night','Late Night']
df['Time_of_Day'] = np.select(conditions, choices, default='Unknown')

# ── Preparation Category ──────────────────────────────────────────────────
df['Prep_Category'] = pd.cut(
    df['Preparation_Time_Min'].astype(float),
    bins=[-np.inf, 10, 20, np.inf],
    labels=['Fast','Moderate','Slow']
).astype(str)

# ── Rider Experience Category ─────────────────────────────────────────────
df['Experience_Category'] = pd.cut(
    df['Rider_Experience_Years'].astype(float),
    bins=[-np.inf, 1, 3, np.inf],
    labels=['Novice','Junior','Senior']
).astype(str)

# ── Speed Category ────────────────────────────────────────────────────────
df['Speed_Category'] = pd.cut(
    df['Average_Speed_kmph'].astype(float),
    bins=[-np.inf, 20, 35, np.inf],
    labels=['Slow','Moderate','Fast']
).astype(str)

# ── Signals Category ─────────────────────────────────────────────────────
df['Signals_Category'] = pd.cut(
    df['Number_of_Signals'].astype(float),
    bins=[-np.inf, 5, 15, np.inf],
    labels=['Low','Medium','High']
).astype(str)

# ── Day sort order (Mon=0 ... Sun=6) ─────────────────────────────────────
DAY_ORDER_MAP = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,
                 'Friday':4,'Saturday':5,'Sunday':6}
df['Day_Order'] = df['Day_of_Week'].str.strip().str.title().map(DAY_ORDER_MAP).fillna(-1).astype(int)

# ── Pickup-Dropoff corridor ───────────────────────────────────────────────
df['Pickup_Dropoff_Pair'] = df['Pickup_Zone'].str.strip() + ' -> ' + df['Dropoff_Zone'].str.strip()

print(f'Feature engineering complete.')
print(f'DataFrame now has {len(df.columns)} columns:')
print([c for c in df.columns if c not in [
    'Order_ID','Order_Date','Order_Hour','Day_of_Week','Is_Weekend',
    'Is_Festival','Weather','Pickup_Zone','Dropoff_Zone','Vehicle_Type',
    'Rider_Experience_Years','Rider_Rating','Restaurant_Rating','Cuisine_Type',
    'Order_Items','Restaurant_Load','Preparation_Time_Min','Road_Distance_km',
    'Delivery_Distance_Category','Traffic_Level','Number_of_Signals',
    'Average_Speed_kmph','Delivery_Priority','Time_taken_min']])

## 6. 10 KPI Calculations

In [ ]:
total_orders     = len(df)
avg_delivery     = round(float(df['Time_taken_min'].mean()), 1)
med_delivery     = round(float(df['Time_taken_min'].median()), 1)
avg_prep         = round(float(df['Preparation_Time_Min'].mean()), 1)
avg_distance     = round(float(df['Road_Distance_km'].mean()), 2)
avg_rider_rating = round(float(df['Rider_Rating'].mean()), 2)
avg_rest_rating  = round(float(df['Restaurant_Rating'].mean()), 2)
avg_speed        = round(float(df['Average_Speed_kmph'].mean()), 1)
weekend_pct      = round(float(df['Is_Weekend'].sum() / total_orders * 100), 1)
festival_pct     = round(float(df['Is_Festival'].sum() / total_orders * 100), 1)

kpis = {
    'KPI': [
        'Total Orders',
        'Avg Delivery Time (min)',
        'Median Delivery Time (min)',
        'Avg Preparation Time (min)',
        'Avg Road Distance (km)',
        'Avg Rider Rating',
        'Avg Restaurant Rating',
        'Avg Speed (km/h)',
        'Weekend Orders (%)',
        'Festival Orders (%)',
    ],
    'Value': [
        f'{total_orders:,}',
        str(avg_delivery),
        str(med_delivery),
        str(avg_prep),
        str(avg_distance),
        str(avg_rider_rating),
        str(avg_rest_rating),
        str(avg_speed),
        f'{weekend_pct}%',
        f'{festival_pct}%',
    ],
}

kpi_df = pd.DataFrame(kpis)
print('=== 10 KEY PERFORMANCE INDICATORS (Computed from Actual Dataset) ===')
print(kpi_df.to_string(index=False))

In [ ]:
# ── Visual KPI bar chart ──────────────────────────────────────────────────
numeric_kpis = {
    'Avg Delivery\nTime (min)'  : avg_delivery,
    'Median Delivery\nTime (min)': med_delivery,
    'Avg Prep\nTime (min)'      : avg_prep,
    'Avg Distance\n(km)'        : avg_distance,
    'Avg Rider\nRating'         : avg_rider_rating,
    'Avg Rest.\nRating'         : avg_rest_rating,
    'Avg Speed\n(km/h)'         : avg_speed,
    'Weekend\nOrders (%)'       : weekend_pct,
    'Festival\nOrders (%)'      : festival_pct,
}

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(numeric_kpis.keys(), numeric_kpis.values(),
              color=PRIMARY, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, numeric_kpis.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('10 Key Performance Indicators — Food Delivery Analytics',
             fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Value')
ax.set_xlabel('KPI')
sns.despine()
plt.tight_layout()
plt.show()

## 7. EDA — Univariate Analysis

In [ ]:
# ── Distribution plots for key numeric columns ────────────────────────────
num_cols = ['Time_taken_min','Road_Distance_km','Preparation_Time_Min',
            'Average_Speed_kmph','Rider_Rating','Restaurant_Rating',
            'Rider_Experience_Years','Number_of_Signals','Order_Items']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = df[col].dropna().astype(float)
    axes[i].hist(data, bins=40, color=PRIMARY, alpha=0.85, edgecolor='white')
    axes[i].axvline(data.mean(), color=ACCENT, linestyle='--', linewidth=1.8,
                    label=f'Mean: {data.mean():.1f}')
    axes[i].axvline(data.median(), color='green', linestyle=':', linewidth=1.8,
                    label=f'Median: {data.median():.1f}')
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=8)

plt.suptitle('Distribution of Numeric Variables', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Frequency bar charts for categorical columns ──────────────────────────
cat_cols_plot = ['Weather','Traffic_Level','Vehicle_Type','Cuisine_Type',
                 'Pickup_Zone','Dropoff_Zone','Delivery_Priority',
                 'Restaurant_Load','Delivery_Distance_Category']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols_plot):
    vc = df[col].value_counts()
    axes[i].barh(vc.index, vc.values, color=PRIMARY, alpha=0.85)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('Count')
    for j, val in enumerate(vc.values):
        axes[i].text(val + 50, j, f'{val:,}', va='center', fontsize=8)

plt.suptitle('Frequency Distribution of Categorical Variables',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. EDA — Bivariate Analysis

In [ ]:
# ── Box plots: Delivery time vs key categorical variables ─────────────────
box_cols = ['Traffic_Level','Weather','Vehicle_Type','Day_of_Week',
            'Delivery_Priority','Restaurant_Load']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(box_cols):
    order = df.groupby(col)['Time_taken_min'].median().sort_values().index.tolist()
    sns.boxplot(data=df, x=col, y='Time_taken_min', order=order,
                palette='Blues', ax=axes[i], showfliers=False)
    axes[i].axhline(avg_delivery, color=ACCENT, linestyle='--',
                    linewidth=1.5, label=f'Overall avg: {avg_delivery} min')
    axes[i].set_title(f'Delivery Time by {col}', fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Delivery Time (min)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(fontsize=8)

plt.suptitle('Delivery Time Distribution by Categorical Variables',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: Road Distance vs Delivery Time (5,000-row sample) ────────────
sample = df.sample(min(5000, len(df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(sample['Road_Distance_km'], sample['Time_taken_min'],
                alpha=0.25, c=PRIMARY, s=10)
z = np.polyfit(sample['Road_Distance_km'].astype(float),
               sample['Time_taken_min'].astype(float), 1)
p = np.poly1d(z)
xs = np.linspace(sample['Road_Distance_km'].min(), sample['Road_Distance_km'].max(), 200)
axes[0].plot(xs, p(xs), color=ACCENT, linewidth=2, label='Trend line')
axes[0].set_title('Road Distance vs Delivery Time\n(5,000 sample)', fontsize=11)
axes[0].set_xlabel('Road Distance (km)')
axes[0].set_ylabel('Delivery Time (min)')
axes[0].legend()

axes[1].scatter(sample['Preparation_Time_Min'], sample['Time_taken_min'],
                alpha=0.25, c='#7c5cd8', s=10)
z2 = np.polyfit(sample['Preparation_Time_Min'].astype(float),
                sample['Time_taken_min'].astype(float), 1)
p2 = np.poly1d(z2)
xs2 = np.linspace(sample['Preparation_Time_Min'].min(), sample['Preparation_Time_Min'].max(), 200)
axes[1].plot(xs2, p2(xs2), color=ACCENT, linewidth=2, label='Trend line')
axes[1].set_title('Preparation Time vs Delivery Time\n(5,000 sample)', fontsize=11)
axes[1].set_xlabel('Preparation Time (min)')
axes[1].set_ylabel('Delivery Time (min)')
axes[1].legend()

plt.suptitle('Scatter Analysis — Key Numeric Relationships', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Correlation Analysis

In [ ]:
# ── Pearson correlation matrix ─────────────────────────────────────────────
corr_cols = ['Time_taken_min','Road_Distance_km','Preparation_Time_Min',
             'Average_Speed_kmph','Rider_Rating','Restaurant_Rating',
             'Rider_Experience_Years','Number_of_Signals','Order_Items',
             'Order_Hour','Is_Weekend','Is_Festival']

corr_df = df[corr_cols].astype(float).corr(method='pearson').round(3)

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(
    corr_df, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8}, ax=ax
)
ax.set_title('Pearson Correlation Heatmap — Numeric Variables',
             fontsize=13, fontweight='bold', pad=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Top correlations with delivery time
print('\nTop correlations with Time_taken_min (delivery time):')
target_corr = corr_df['Time_taken_min'].drop('Time_taken_min').sort_values(key=abs, ascending=False)
print(target_corr.to_string())

## 10. Order Trend Analysis

In [ ]:
# ── Daily order volume ────────────────────────────────────────────────────
daily_orders = (df.groupby('Order_Date').size()
                  .reset_index(name='Order_Count')
                  .sort_values('Order_Date'))

# ── Daily average delivery time ───────────────────────────────────────────
daily_time = (df.groupby('Order_Date')['Time_taken_min']
                .mean().round(2)
                .reset_index()
                .rename(columns={'Time_taken_min': 'Avg_Delivery_Time'})
                .sort_values('Order_Date'))

fig, axes = plt.subplots(2, 1, figsize=(15, 9))

axes[0].plot(daily_orders['Order_Date'], daily_orders['Order_Count'],
             color=PRIMARY, linewidth=1.5)
axes[0].fill_between(daily_orders['Order_Date'], daily_orders['Order_Count'],
                     alpha=0.15, color=PRIMARY)
axes[0].set_title('Daily Order Volume Over Time', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Orders')
axes[0].set_xlabel('')

axes[1].plot(daily_time['Order_Date'], daily_time['Avg_Delivery_Time'],
             color='#7c5cd8', linewidth=1.5)
axes[1].axhline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5,
                label=f'Overall avg: {avg_delivery} min')
axes[1].set_title('Daily Average Delivery Time Trend', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Avg Delivery Time (min)')
axes[1].set_xlabel('Date')
axes[1].legend()

plt.tight_layout()
plt.show()

# Monthly breakdown
monthly = df.groupby('Month').size().reset_index(name='Order_Count')
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(monthly['Month'], monthly['Order_Count'], color=PRIMARY, edgecolor='white')
ax.set_title('Monthly Order Volume', fontsize=12, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Orders')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

## 11. BQ1 — What is the average delivery time across all orders?

In [ ]:
bq1 = {
    'mean'  : round(float(df['Time_taken_min'].mean()),  2),
    'median': round(float(df['Time_taken_min'].median()), 2),
    'std'   : round(float(df['Time_taken_min'].std()),   2),
    'min'   : int(df['Time_taken_min'].min()),
    'max'   : int(df['Time_taken_min'].max()),
}

print('BQ1: Average Delivery Time')
for k, v in bq1.items():
    print(f'  {k:8s} : {v}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with mean/median
axes[0].hist(df['Time_taken_min'].astype(float), bins=50,
             color=PRIMARY, edgecolor='white', alpha=0.85)
axes[0].axvline(bq1['mean'],   color=ACCENT,   linestyle='--', linewidth=2,
                label=f"Mean: {bq1['mean']} min")
axes[0].axvline(bq1['median'], color='green',  linestyle=':',  linewidth=2,
                label=f"Median: {bq1['median']} min")
axes[0].set_title('Delivery Time Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Delivery Time (min)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot
axes[1].boxplot(df['Time_taken_min'].dropna().astype(float),
                patch_artist=True,
                boxprops=dict(facecolor='#bfd7f5', color=PRIMARY),
                medianprops=dict(color=ACCENT, linewidth=2))
axes[1].set_title('Delivery Time Box Plot', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Delivery Time (min)')
axes[1].set_xticks([])

plt.suptitle(f'BQ1: Avg={bq1["mean"]} min | Median={bq1["median"]} min | '
             f'Range=[{bq1["min"]}, {bq1["max"]}] min',
             fontsize=12)
plt.tight_layout()
plt.show()

## 12. BQ2 — Which day of the week has the highest average delivery time?

In [ ]:
day_order_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,
                 'Friday':4,'Saturday':5,'Sunday':6}

bq2 = (df.groupby('Day_of_Week')['Time_taken_min']
         .agg(Mean='mean', Median='median', Std='std', Count='count')
         .round(2).reset_index())
bq2['Day_Order'] = bq2['Day_of_Week'].map(day_order_map).fillna(7)
bq2 = bq2.sort_values('Day_Order').drop(columns='Day_Order')

print('BQ2: Average Delivery Time by Day of Week')
print(bq2.to_string(index=False))
print(f"\nHighest: {bq2.iloc[bq2['Mean'].argmax()]['Day_of_Week']} ({bq2['Mean'].max():.1f} min)")
print(f"Lowest : {bq2.iloc[bq2['Mean'].argmin()]['Day_of_Week']} ({bq2['Mean'].min():.1f} min)")

fig, ax = plt.subplots(figsize=(11, 5))
colors = [ACCENT if v == bq2['Mean'].max() else PRIMARY for v in bq2['Mean']]
bars = ax.bar(bq2['Day_of_Week'], bq2['Mean'], color=colors, edgecolor='white')
ax.axhline(avg_delivery, color='grey', linestyle='--', linewidth=1.5,
           label=f'Overall avg: {avg_delivery} min')
for bar, val in zip(bars, bq2['Mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('BQ2: Average Delivery Time by Day of Week', fontsize=13, fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Avg Delivery Time (min)')
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

## 13. BQ3 — Which time of day experiences the highest delivery delays?

In [ ]:
tod_order = ['Morning','Afternoon','Evening','Night','Late Night']

bq3 = (df.groupby('Time_of_Day')['Time_taken_min']
         .agg(Mean='mean', Median='median', Count='count')
         .round(2).reset_index())
bq3['sort'] = bq3['Time_of_Day'].map({t:i for i,t in enumerate(tod_order)}).fillna(99)
bq3 = bq3.sort_values('sort').drop(columns='sort')

print('BQ3: Delivery Time by Time of Day')
print(bq3.to_string(index=False))
worst_tod = bq3.loc[bq3['Mean'].idxmax(), 'Time_of_Day']
print(f"\nHighest delay period: {worst_tod} ({bq3['Mean'].max():.1f} min avg)")

# Also by hour
hourly = (df.groupby('Order_Hour')['Time_taken_min']
            .mean().round(2).reset_index()
            .rename(columns={'Time_taken_min': 'Avg'})
            .sort_values('Order_Hour'))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors_tod = [ACCENT if v == bq3['Mean'].max() else PRIMARY for v in bq3['Mean']]
axes[0].bar(bq3['Time_of_Day'], bq3['Mean'], color=colors_tod, edgecolor='white')
axes[0].axhline(avg_delivery, color='grey', linestyle='--', linewidth=1.5,
                label=f'Overall avg: {avg_delivery} min')
axes[0].set_title('BQ3: Avg Delivery Time by Time of Day', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Time of Day')
axes[0].set_ylabel('Avg Delivery Time (min)')
axes[0].legend()

axes[1].bar(hourly['Order_Hour'].astype(str), hourly['Avg'],
            color=PRIMARY, edgecolor='white')
axes[1].axhline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5,
                label=f'Overall avg: {avg_delivery} min')
axes[1].set_title('Avg Delivery Time by Hour of Day (0-23)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Avg Delivery Time (min)')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=90)

sns.despine()
plt.tight_layout()
plt.show()

## 14. BQ4 — How does traffic level affect delivery time?

In [ ]:
bq4 = (df.groupby('Traffic_Level')['Time_taken_min']
         .agg(Mean='mean', Median='median', Std='std', Count='count')
         .round(2).reset_index())
bq4['Pct_Above_Overall'] = ((bq4['Mean'] - avg_delivery) / avg_delivery * 100).round(1)
bq4 = bq4.sort_values('Mean', ascending=False)

print('BQ4: Delivery Time by Traffic Level')
print(bq4.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box plot
order_traffic = bq4['Traffic_Level'].tolist()
sns.boxplot(data=df, x='Traffic_Level', y='Time_taken_min',
            order=order_traffic, palette='Blues_r', ax=axes[0], showfliers=False)
axes[0].axhline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5,
                label=f'Overall avg: {avg_delivery} min')
axes[0].set_title('Delivery Time Distribution by Traffic Level', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Traffic Level')
axes[0].set_ylabel('Delivery Time (min)')
axes[0].legend()

# % above overall average
colors_t = [ACCENT if v > 0 else '#5aba7a' for v in bq4['Pct_Above_Overall']]
axes[1].bar(bq4['Traffic_Level'], bq4['Pct_Above_Overall'], color=colors_t, edgecolor='white')
axes[1].axhline(0, color='grey', linewidth=1)
for bar, val in zip(axes[1].patches, bq4['Pct_Above_Overall']):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5 if val >= 0 else bar.get_height() - 1.5,
                 f'{val:+.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_title('% Difference from Overall Avg by Traffic Level',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Traffic Level')
axes[1].set_ylabel('% vs Overall Avg')

sns.despine()
plt.tight_layout()
plt.show()

## 15. BQ5 — How does road distance affect delivery time?

In [ ]:
dist_order = ['Short','Medium','Long']
bq5 = (df.groupby('Delivery_Distance_Category')['Time_taken_min']
         .agg(Mean='mean', Median='median', Std='std', Count='count')
         .round(2).reset_index())
bq5['sort'] = bq5['Delivery_Distance_Category'].map({d:i for i,d in enumerate(dist_order)}).fillna(99)
bq5 = bq5.sort_values('sort').drop(columns='sort')

print('BQ5: Delivery Time by Distance Category')
print(bq5.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(bq5['Delivery_Distance_Category'], bq5['Mean'],
            color=[PRIMARY,'#5b9bd5','#2e75b6'], edgecolor='white')
axes[0].axhline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5,
                label=f'Overall avg: {avg_delivery} min')
for bar, val in zip(axes[0].patches, bq5['Mean']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('BQ5: Avg Delivery Time by Distance Category', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Distance Category')
axes[0].set_ylabel('Avg Delivery Time (min)')
axes[0].legend()

smp = df.sample(min(4000, len(df)), random_state=42)
for cat, color in zip(dist_order, [PRIMARY,'#5b9bd5','#2e75b6']):
    sub = smp[smp['Delivery_Distance_Category'] == cat]
    axes[1].scatter(sub['Road_Distance_km'], sub['Time_taken_min'],
                    alpha=0.3, s=8, label=cat, c=color)
axes[1].set_title('Road Distance vs Delivery Time (by category)',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Road Distance (km)')
axes[1].set_ylabel('Delivery Time (min)')
axes[1].legend()

sns.despine()
plt.tight_layout()
plt.show()

## 16. BQ6 — Which vehicle type has the lowest average delivery time?

In [ ]:
bq6 = (df.groupby('Vehicle_Type')['Time_taken_min']
         .agg(Mean='mean', Median='median', Std='std', Count='count')
         .round(2).reset_index()
         .sort_values('Mean'))

print('BQ6: Delivery Time by Vehicle Type')
print(bq6.to_string(index=False))
print(f"\nFastest: {bq6.iloc[0]['Vehicle_Type']} ({bq6.iloc[0]['Mean']:.1f} min avg)")
print(f"Slowest: {bq6.iloc[-1]['Vehicle_Type']} ({bq6.iloc[-1]['Mean']:.1f} min avg)")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors_v = [ACCENT if i == 0 else PRIMARY for i in range(len(bq6))]
axes[0].barh(bq6['Vehicle_Type'], bq6['Mean'], color=colors_v, edgecolor='white')
axes[0].axvline(avg_delivery, color='grey', linestyle='--', linewidth=1.5,
                label=f'Overall avg: {avg_delivery} min')
for bar, val in zip(axes[0].patches, bq6['Mean']):
    axes[0].text(val + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9, fontweight='bold')
axes[0].set_title('BQ6: Avg Delivery Time by Vehicle Type', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Avg Delivery Time (min)')
axes[0].legend()

sns.boxplot(data=df, x='Vehicle_Type', y='Time_taken_min',
            order=bq6['Vehicle_Type'].tolist(), palette='Blues',
            ax=axes[1], showfliers=False)
axes[1].axhline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5)
axes[1].set_title('Delivery Time Distribution by Vehicle Type', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Vehicle Type')
axes[1].set_ylabel('Delivery Time (min)')
axes[1].tick_params(axis='x', rotation=20)

sns.despine()
plt.tight_layout()
plt.show()

## 17. BQ7 — How does restaurant preparation time affect total delivery time?

In [ ]:
prep_order = ['Fast','Moderate','Slow']
bq7 = (df.groupby('Prep_Category')['Time_taken_min']
         .agg(Mean='mean', Median='median', Std='std', Count='count')
         .round(2).reset_index())
bq7['sort'] = bq7['Prep_Category'].map({d:i for i,d in enumerate(prep_order)}).fillna(99)
bq7 = bq7.sort_values('sort').drop(columns='sort')

prep_corr = round(float(
    df[['Preparation_Time_Min','Time_taken_min']].astype(float)
    .corr().loc['Preparation_Time_Min','Time_taken_min']), 3)

print('BQ7: Delivery Time by Preparation Category')
print(bq7.to_string(index=False))
print(f'\nPearson correlation (Prep Time vs Delivery Time): {prep_corr}')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(bq7['Prep_Category'], bq7['Mean'],
            color=[PRIMARY,'#5b9bd5','#2e75b6'], edgecolor='white')
axes[0].axhline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5,
                label=f'Overall avg: {avg_delivery} min')
for bar, val in zip(axes[0].patches, bq7['Mean']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('BQ7: Avg Delivery Time by Preparation Speed', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Preparation Category')
axes[0].set_ylabel('Avg Delivery Time (min)')
axes[0].legend()

smp2 = df.sample(min(5000, len(df)), random_state=42)
axes[1].scatter(smp2['Preparation_Time_Min'], smp2['Time_taken_min'],
                alpha=0.25, c=PRIMARY, s=10)
z3 = np.polyfit(smp2['Preparation_Time_Min'].astype(float),
                smp2['Time_taken_min'].astype(float), 1)
xs3 = np.linspace(smp2['Preparation_Time_Min'].min(), smp2['Preparation_Time_Min'].max(), 200)
axes[1].plot(xs3, np.poly1d(z3)(xs3), color=ACCENT, linewidth=2,
             label=f'Trend (r={prep_corr})')
axes[1].set_title('Preparation Time vs Total Delivery Time\n(5,000 sample)',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Preparation Time (min)')
axes[1].set_ylabel('Delivery Time (min)')
axes[1].legend()

sns.despine()
plt.tight_layout()
plt.show()

## 18. BQ8 — Rider Experience & Rating vs Delivery Performance

In [ ]:
exp_order = ['Novice','Junior','Senior']
bq8_exp = (df.groupby('Experience_Category')['Time_taken_min']
             .agg(Mean='mean', Median='median', Count='count')
             .round(2).reset_index())
bq8_exp['sort'] = bq8_exp['Experience_Category'].map({d:i for i,d in enumerate(exp_order)}).fillna(99)
bq8_exp = bq8_exp.sort_values('sort').drop(columns='sort')

rider_corr = round(float(
    df[['Rider_Rating','Time_taken_min']].astype(float)
    .corr().loc['Rider_Rating','Time_taken_min']), 3)

print('BQ8: Delivery Time by Rider Experience Category')
print(bq8_exp.to_string(index=False))
print(f'\nPearson correlation (Rider Rating vs Delivery Time): {rider_corr}')
direction = 'negative' if rider_corr < 0 else 'positive'
print(f'Interpretation: {direction} correlation — higher rating -> '
      f'{'shorter' if rider_corr < 0 else 'longer'} delivery time')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(bq8_exp['Experience_Category'], bq8_exp['Mean'],
            color=[PRIMARY,'#5b9bd5','#2e75b6'], edgecolor='white')
axes[0].axhline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5,
                label=f'Overall avg: {avg_delivery} min')
for bar, val in zip(axes[0].patches, bq8_exp['Mean']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Avg Delivery Time by Rider Experience', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Experience Category')
axes[0].set_ylabel('Avg Delivery Time (min)')
axes[0].legend()

smp3 = df.sample(min(5000, len(df)), random_state=42)
axes[1].scatter(smp3['Rider_Rating'], smp3['Time_taken_min'],
                alpha=0.25, c=PRIMARY, s=15)
z4 = np.polyfit(smp3['Rider_Rating'].astype(float),
                smp3['Time_taken_min'].astype(float), 1)
xs4 = np.linspace(smp3['Rider_Rating'].min(), smp3['Rider_Rating'].max(), 100)
axes[1].plot(xs4, np.poly1d(z4)(xs4), color=ACCENT, linewidth=2,
             label=f'Trend (r={rider_corr})')
axes[1].set_title('Rider Rating vs Delivery Time\n(5,000 sample)',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Rider Rating')
axes[1].set_ylabel('Delivery Time (min)')
axes[1].legend()

sns.despine()
plt.tight_layout()
plt.show()

## 19. BQ9 — How do weather conditions affect delivery time?

In [ ]:
bq9 = (df.groupby('Weather')['Time_taken_min']
         .agg(Mean='mean', Median='median', Std='std', Count='count')
         .round(2).reset_index())
bq9['Pct_Above_Overall'] = ((bq9['Mean'] - avg_delivery) / avg_delivery * 100).round(1)
bq9 = bq9.sort_values('Mean', ascending=False)

print('BQ9: Delivery Time by Weather Condition')
print(bq9.to_string(index=False))
print(f"\nWorst weather : {bq9.iloc[0]['Weather']} ({bq9.iloc[0]['Pct_Above_Overall']:+.1f}% vs overall)")
print(f"Best weather  : {bq9.iloc[-1]['Weather']} ({bq9.iloc[-1]['Pct_Above_Overall']:+.1f}% vs overall)")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=df, x='Weather', y='Time_taken_min',
            order=bq9['Weather'].tolist(), palette='Blues_r',
            ax=axes[0], showfliers=False)
axes[0].axhline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5,
                label=f'Overall avg: {avg_delivery} min')
axes[0].set_title('BQ9: Delivery Time by Weather Condition', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Weather')
axes[0].set_ylabel('Delivery Time (min)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend()

colors_w = [ACCENT if v > 0 else '#5aba7a' for v in bq9['Pct_Above_Overall']]
axes[1].barh(bq9['Weather'], bq9['Pct_Above_Overall'], color=colors_w, edgecolor='white')
axes[1].axvline(0, color='grey', linewidth=1)
for bar, val in zip(axes[1].patches, bq9['Pct_Above_Overall']):
    axes[1].text(val + 0.3 if val >= 0 else val - 0.3,
                 bar.get_y() + bar.get_height()/2,
                 f'{val:+.1f}%', va='center', fontsize=9, fontweight='bold')
axes[1].set_title('% Difference from Overall Avg by Weather', fontsize=11, fontweight='bold')
axes[1].set_xlabel('% vs Overall Avg')

sns.despine()
plt.tight_layout()
plt.show()

## 20. BQ10 — Which pickup and drop-off zones have the highest average delivery time?

In [ ]:
pickup_stats = (df.groupby('Pickup_Zone')['Time_taken_min']
                  .mean().round(2).reset_index()
                  .rename(columns={'Time_taken_min': 'Avg_Delivery_Time'})
                  .sort_values('Avg_Delivery_Time', ascending=False))

dropoff_stats = (df.groupby('Dropoff_Zone')['Time_taken_min']
                   .mean().round(2).reset_index()
                   .rename(columns={'Time_taken_min': 'Avg_Delivery_Time'})
                   .sort_values('Avg_Delivery_Time', ascending=False))

pairs_stats = (df.groupby('Pickup_Dropoff_Pair')['Time_taken_min']
                 .mean().round(2).reset_index()
                 .rename(columns={'Time_taken_min': 'Avg_Delivery_Time'})
                 .sort_values('Avg_Delivery_Time', ascending=False)
                 .head(10))

print('BQ10: Pickup Zone Performance')
print(pickup_stats.to_string(index=False))
print('\nBQ10: Drop-off Zone Performance')
print(dropoff_stats.to_string(index=False))
print('\nTop 10 Slowest Corridors:')
print(pairs_stats.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].barh(pickup_stats['Pickup_Zone'],
             pickup_stats['Avg_Delivery_Time'],
             color=PRIMARY, edgecolor='white')
axes[0].axvline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5)
axes[0].set_title('BQ10: Avg Delivery Time by Pickup Zone', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Avg Delivery Time (min)')

axes[1].barh(dropoff_stats['Dropoff_Zone'],
             dropoff_stats['Avg_Delivery_Time'],
             color='#7c5cd8', edgecolor='white')
axes[1].axvline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5)
axes[1].set_title('BQ10: Avg Delivery Time by Drop-off Zone', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Avg Delivery Time (min)')

sns.despine()
plt.tight_layout()
plt.show()

# Pickup x Dropoff heatmap
pivot = df.pivot_table(values='Time_taken_min', index='Pickup_Zone',
                       columns='Dropoff_Zone', aggfunc='mean').round(1)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Delivery Time (min)'})
ax.set_title('Avg Delivery Time: Pickup Zone x Drop-off Zone (minutes)',
             fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

## 21. Operational & Environmental Analysis

In [ ]:
# ── Weekend vs Weekday ────────────────────────────────────────────────────
weekend_comp = (df.groupby('Is_Weekend')['Time_taken_min']
                  .mean().round(2).reset_index())
weekend_comp['Label'] = weekend_comp['Is_Weekend'].map({0: 'Weekday', 1: 'Weekend'})

# ── Festival vs Non-festival ──────────────────────────────────────────────
festival_comp = (df.groupby('Is_Festival')['Time_taken_min']
                   .mean().round(2).reset_index())
festival_comp['Label'] = festival_comp['Is_Festival'].map({0: 'Non-Festival', 1: 'Festival'})

# ── Delivery Priority ─────────────────────────────────────────────────────
priority_stats = (df.groupby('Delivery_Priority')['Time_taken_min']
                    .mean().round(2).reset_index()
                    .rename(columns={'Time_taken_min': 'Avg'})
                    .sort_values('Avg', ascending=False))

# ── Restaurant Load ───────────────────────────────────────────────────────
load_stats = (df.groupby('Restaurant_Load')['Time_taken_min']
                .mean().round(2).reset_index()
                .rename(columns={'Time_taken_min': 'Avg'})
                .sort_values('Avg', ascending=False))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Weekend
bars0 = axes[0,0].bar(weekend_comp['Label'], weekend_comp['Time_taken_min'],
                       color=[PRIMARY, ACCENT], edgecolor='white', width=0.5)
axes[0,0].axhline(avg_delivery, color='grey', linestyle='--', linewidth=1.5)
for bar, val in zip(bars0, weekend_comp['Time_taken_min']):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                   f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')
axes[0,0].set_title('Weekend vs Weekday Delivery Time', fontsize=11, fontweight='bold')
axes[0,0].set_ylabel('Avg Delivery Time (min)')

# Festival
bars1 = axes[0,1].bar(festival_comp['Label'], festival_comp['Time_taken_min'],
                       color=[PRIMARY, ACCENT], edgecolor='white', width=0.5)
axes[0,1].axhline(avg_delivery, color='grey', linestyle='--', linewidth=1.5)
for bar, val in zip(bars1, festival_comp['Time_taken_min']):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                   f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')
axes[0,1].set_title('Festival vs Non-Festival Delivery Time', fontsize=11, fontweight='bold')
axes[0,1].set_ylabel('Avg Delivery Time (min)')

# Priority
axes[1,0].barh(priority_stats['Delivery_Priority'], priority_stats['Avg'],
               color=PRIMARY, edgecolor='white')
axes[1,0].axvline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5)
axes[1,0].set_title('Delivery Time by Priority Level', fontsize=11, fontweight='bold')
axes[1,0].set_xlabel('Avg Delivery Time (min)')

# Restaurant Load
axes[1,1].barh(load_stats['Restaurant_Load'], load_stats['Avg'],
               color='#7c5cd8', edgecolor='white')
axes[1,1].axvline(avg_delivery, color=ACCENT, linestyle='--', linewidth=1.5)
axes[1,1].set_title('Delivery Time by Restaurant Load', fontsize=11, fontweight='bold')
axes[1,1].set_xlabel('Avg Delivery Time (min)')

plt.suptitle('Operational & Environmental Analysis', fontsize=14, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ── Cuisine analysis ──────────────────────────────────────────────────────
cuisine_orders = (df.groupby('Cuisine_Type').size()
                    .reset_index(name='Order_Count')
                    .sort_values('Order_Count', ascending=False))

cuisine_prep = (df.groupby('Cuisine_Type')['Preparation_Time_Min']
                  .mean().round(2).reset_index()
                  .rename(columns={'Preparation_Time_Min': 'Avg_Prep'})
                  .sort_values('Avg_Prep', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].barh(cuisine_orders['Cuisine_Type'], cuisine_orders['Order_Count'],
             color=PRIMARY, edgecolor='white')
axes[0].set_title('Order Count by Cuisine Type', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Number of Orders')

axes[1].barh(cuisine_prep['Cuisine_Type'], cuisine_prep['Avg_Prep'],
             color='#7c5cd8', edgecolor='white')
axes[1].set_title('Avg Preparation Time by Cuisine Type', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Avg Preparation Time (min)')

sns.despine()
plt.tight_layout()
plt.show()

## 22. Business Insights Summary

In [ ]:
# ── All insights computed programmatically from actual data ───────────────
print('=' * 65)
print('  FOOD DELIVERY ANALYTICS — BUSINESS INSIGHTS SUMMARY')
print('  IBM Data Analytics Internship | Khushi Chaudhary')
print('=' * 65)

print('\n[SECTION 1: KEY PERFORMANCE INDICATORS]')
print(f'  Total Orders          : {total_orders:,}')
print(f'  Avg Delivery Time     : {avg_delivery} min')
print(f'  Median Delivery Time  : {med_delivery} min')
print(f'  Avg Preparation Time  : {avg_prep} min')
print(f'  Avg Road Distance     : {avg_distance} km')
print(f'  Avg Rider Rating      : {avg_rider_rating} / 5.0')
print(f'  Avg Restaurant Rating : {avg_rest_rating} / 5.0')
print(f'  Avg Speed             : {avg_speed} km/h')
print(f'  Weekend Orders        : {weekend_pct}%')
print(f'  Festival Orders       : {festival_pct}%')

print('\n[SECTION 2: 10 BUSINESS QUESTIONS — ANSWERS]')

# BQ1
print(f'\n  BQ1 - Average Delivery Time:')
print(f'    Mean={bq1["mean"]} min | Median={bq1["median"]} min | '
      f'Range=[{bq1["min"]},{bq1["max"]}] min')

# BQ2
worst_day_row = bq2.loc[bq2['Mean'].idxmax()]
best_day_row  = bq2.loc[bq2['Mean'].idxmin()]
print(f'\n  BQ2 - Day of Week:')
print(f'    Highest: {worst_day_row["Day_of_Week"]} ({worst_day_row["Mean"]:.1f} min)')
print(f'    Lowest : {best_day_row["Day_of_Week"]} ({best_day_row["Mean"]:.1f} min)')

# BQ3
worst_tod_row = bq3.loc[bq3['Mean'].idxmax()]
print(f'\n  BQ3 - Time of Day:')
print(f'    Highest delay period: {worst_tod_row["Time_of_Day"]} ({worst_tod_row["Mean"]:.1f} min avg)')

# BQ4
print(f'\n  BQ4 - Traffic Impact:')
for _, row in bq4.iterrows():
    print(f'    {row["Traffic_Level"]:15s}: {row["Mean"]:.1f} min ({row["Pct_Above_Overall"]:+.1f}% vs overall)')

# BQ5
print(f'\n  BQ5 - Distance Impact:')
for _, row in bq5.iterrows():
    print(f'    {row["Delivery_Distance_Category"]:10s}: {row["Mean"]:.1f} min avg')

# BQ6
print(f'\n  BQ6 - Vehicle Type:')
print(f'    Fastest: {bq6.iloc[0]["Vehicle_Type"]} ({bq6.iloc[0]["Mean"]:.1f} min avg)')
print(f'    Slowest: {bq6.iloc[-1]["Vehicle_Type"]} ({bq6.iloc[-1]["Mean"]:.1f} min avg)')

# BQ7
print(f'\n  BQ7 - Preparation Time:')
print(f'    Pearson r (Prep vs Delivery): {prep_corr}')
for _, row in bq7.iterrows():
    print(f'    {row["Prep_Category"]:10s}: {row["Mean"]:.1f} min avg delivery')

# BQ8
print(f'\n  BQ8 - Rider Performance:')
print(f'    Rider Rating Pearson r vs Delivery Time: {rider_corr}')
direction = 'negative' if rider_corr < 0 else 'positive'
print(f'    Correlation direction: {direction}')
for _, row in bq8_exp.iterrows():
    print(f'    {row["Experience_Category"]:10s}: {row["Mean"]:.1f} min avg delivery')

# BQ9
print(f'\n  BQ9 - Weather Impact:')
for _, row in bq9.iterrows():
    print(f'    {row["Weather"]:15s}: {row["Mean"]:.1f} min ({row["Pct_Above_Overall"]:+.1f}% vs overall)')

# BQ10
print(f'\n  BQ10 - Zone Performance:')
print(f'    Slowest Pickup Zone  : {pickup_stats.iloc[0]["Pickup_Zone"]} ({pickup_stats.iloc[0]["Avg_Delivery_Time"]:.1f} min)')
print(f'    Slowest Dropoff Zone : {dropoff_stats.iloc[0]["Dropoff_Zone"]} ({dropoff_stats.iloc[0]["Avg_Delivery_Time"]:.1f} min)')
print(f'    Slowest Corridor     : {pairs_stats.iloc[0]["Pickup_Dropoff_Pair"]} ({pairs_stats.iloc[0]["Avg_Delivery_Time"]:.1f} min)')

print('\n' + '=' * 65)
print('  ANALYSIS COMPLETE — No ML | No GenAI | Pure Data Analytics')
print('=' * 65)

---

## Project Completion Statement

This notebook contains the **complete end-to-end data analytics pipeline** for the Food Delivery Performance & Operational Analytics project.

| Component | Status |
|-----------|--------|
| Dataset Loading & Validation | Complete |
| Data Cleaning (8 steps) | Complete |
| Data Quality Checks | Complete |
| Feature Engineering (9 columns) | Complete |
| 10 KPIs Calculated | Complete |
| EDA — Univariate Analysis | Complete |
| EDA — Bivariate Analysis | Complete |
| Correlation Analysis | Complete |
| Order Trend Analysis | Complete |
| BQ1–BQ10 All Answered | Complete |
| Operational & Environmental Analysis | Complete |
| Business Insights Summary | Complete |

**No Machine Learning. No Generative AI. No Prediction Models.**  
All insights are computed programmatically from the actual dataset.

---
*Submitted by: Khushi Chaudhary | IBM Data Analytics Internship*